- do any despeckling, do it before converting to dB
- for hackathon, run in the unstable sandbox, as the packages there are the up to date ones that will work with the SAR stuff

In [ ]:
%pip uninstall odc-stac -y
%pip install odc-stac -q
%pip uninstall pystac-client -y
%pip install pystac-client -q
%pip install -U pystac-client -q
%pip uninstall dea-tools -y
%pip install dea-tools==0.4.4 -q
%pip install scikit-image -q


# Land cover classifier using SAR

## Description

1. Load and pre-process Sentinel-1 SAR data
2. Apply a PCA to the processed SAR data
3. Load DEA Land Cover 2.0 and collect training data (alternatively, load DEA Mangroves)
4. Train Random Forest Classifier

---

### Load packages

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import odc.geo.xr
import datacube
import gc

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split,
    StratifiedShuffleSplit,
    RandomizedSearchCV,
    GridSearchCV,
)
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_curve,
    auc,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    average_precision_score,
)
from scipy.stats import randint
from joblib import dump
from matplotlib.colors import ListedColormap
from odc.geo import BoundingBox
from odc.geo.geom import Geometry

from dea_tools.dask import create_local_dask_cluster

# for loading landsat

from dea_tools.datahandling import load_ard
from dea_tools.plotting import rgb
from dea_tools.classification import sklearn_flatten, sklearn_unflatten
from dea_tools.validation import xr_random_sampling
from dea_tools.landcover import lc_colourmap, get_colour_scheme, make_colourbar


In [ ]:
client = create_local_dask_cluster(return_client=True)


In [ ]:
# becaause I'm working in the devbox with a sandbox kernel I can't "find" the src folder so I've copied the speckle filter code here


# Adapted from https://stackoverflow.com/questions/39785970/speckle-lee-filter-in-python
def lee_filter(img, size):
    """
    Applies the Lee filter to reduce speckle noise in an image.

    Parameters:
    img (ndarray): Input image to be filtered.
    size (int): Size of the uniform filter window.

    Returns:
    ndarray: The filtered image.
    """
    img_mean = uniform_filter(img, size)
    img_sqr_mean = uniform_filter(img**2, size)
    img_variance = img_sqr_mean - img_mean**2

    overall_variance = np.var(img)

    img_weights = img_variance / (img_variance + overall_variance)
    img_output = img_mean + img_weights * (img - img_mean)
    return img_output


# Define a function to apply the Lee filter to a DataArray
def apply_lee_filter(data_array, size=7):
    """
    Applies the Lee filter to the provided DataArray.

    Parameters:
    data_array (xarray.DataArray): The data array to be filtered.
    size (int): Size of the uniform filter window. Default is 7.

    Returns:
    xarray.DataArray: The filtered data array.
    """
    data_array_filled = data_array.fillna(0)
    filtered_data = xr.apply_ufunc(
        lee_filter,
        data_array_filled,
        kwargs={"size": size},
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        dask_gufunc_kwargs={"allow_rechunk": True},
        vectorize=True,
        dask="parallelized",
        output_dtypes=[data_array.dtype],
    )
    return filtered_data


### Environment setup

In [ ]:
dc = datacube.Datacube(env="dev", app="S1_Backscatter_Demo")


### Set search parameters

In [ ]:
product_to_load = "ga_s1_nrb_iw_vv_vh_0"
measurements_to_load = ["VV_gamma0", "VH_gamma0", "mask"]
output_crs = "EPSG:3577"
output_res = 20

# region code for mini tiles
region_codes = ["x148y166"]

year = "2024"
start_date = f"{year}-01-01"
end_date = f"{year}-12-30"


In [ ]:
# open tiles and select

gdf = gpd.read_file(
    "~/gdata1/projects/fc-sub-annual/data/testing_minitile_suite.geojson"
)

gdf = gdf[gdf["region_code"].isin(region_codes)]
geom = Geometry(geom=gdf.iloc[0].geometry, crs=gdf.crs)

# for stac:
minx, miny, maxx, maxy = gdf.total_bounds
bbox = [minx, miny, maxx, maxy]
geom_bbox = BoundingBox(left=minx, bottom=miny, right=maxx, top=maxy, crs="EPSG:4326")


### Load Sentinel-1, Sentinel-2 and Land Cover data from the datacube

 * Sentinel-1 SAR data
 * Sentinel-2 multispectral data (for making RGB images of the AOI, maybe also to use as input to model)
 * Land Cover will be used to collect training data for the model

In [ ]:
# using datacube to load multispec data so I can use load_ard for masking
s2 = load_ard(
    dc=dc,
    products=["ga_s2am_ard_3", "ga_s2bm_ard_3"],
    measurements=["nbart_red", "nbart_green", "nbart_blue", "nbart_nir_1"],
    cloud_mask="s2cloudless",
    mask_pixel_quality=True,
    mask_contiguity=True,
    skip_broken_datasets=True,
    time=(start_date, end_date),
    resolution=(-20, 20),
    geopolygon=geom_bbox.boundary(),
    output_crs="EPSG:3577",
    dask_chunks={},
)


In [ ]:
# load SAR with datacube
s1 = dc.load(
    product=product_to_load,
    measurements=measurements_to_load,
    time=(start_date, end_date),
    resolution=(-20, 20),
    output_crs="EPSG:3577",
    geopolygon=geom_bbox.boundary(),
    group_by="solar_day",
    dask_chunks={},
)


In [ ]:
lc = dc.load(
    product="ga_ls_landcover_class_cyear_3",
    measurements=["level3"],
    resolution=(-30, 30),  # make this a multiple of 20
    output_crs="EPSG:3577",
    group_by="solar_day",
    time=(year),
    geopolygon=geom_bbox.boundary(),
    dask_chunks={},
)


### Speckle filtering (optional) and compositing a monthly mean

based on some discussion in Teams, mean is better for SAR than median, and it should be done before converting to db. This may also mean I don't need to do a speckle filter.
Will apply the mask first, and then compare speckly filter and monthly means.

In [ ]:
s1["VV_gamma0_masked"] = xr.where(s1.mask == 0, s1["VV_gamma0"], np.nan)
s1["VH_gamma0_masked"] = xr.where(s1.mask == 0, s1["VH_gamma0"], np.nan)


#### Masking

The GA Sentinel-1 backscatter product comes with a mask that indicates invalid pixels, along with pixels impacted by layover and shadow. 

The masks have the following values:
| Value | Property |
| --- | ----------- |
| 0 | Valid | 
| 1 | Shadow |
| 2 | Layover |
| 3 | Shadow and layover |
| 255 / NaN | Invalid |

The following code displays the masks and shows how to apply them.

In [ ]:
s1["VV_gamma0_masked_filtered"] = apply_lee_filter(s1["VV_gamma0_masked"], size=5)
s1["VH_gamma0_masked_filtered"] = apply_lee_filter(s1["VH_gamma0_masked"], size=5)


## Ratio of VV to VH band

We can calculate the ratio of the vertical polarisation band to the horizontal polarisation band.

This is something done in the DEAfrica notebook [Sentinel-1 Monthly Mosaic](https://docs.digitalearthafrica.org/en/latest/sandbox/notebooks/Datasets/Sentinel_1_Mosaic.html)

Note that because we want to preserve both the monthly means with and without the speckle filtering applied, we are applying the following operations to both arrays.

In [ ]:
# make ratio of VV to VH band
s1["VH_over_VV_gamma0_masked"] = s1["VH_gamma0_masked"] / s1["VV_gamma0_masked"]
s1["VH_over_VV_gamma0_masked_filtered"] = (
    s1["VH_gamma0_masked_filtered"] / s1["VV_gamma0_masked_filtered"]
)


In [ ]:
# converting to dB, simplify data array name for future steps

s1["VV"] = 10 * np.log10(s1["VV_gamma0_masked"])
s1["VH"] = 10 * np.log10(s1["VH_gamma0_masked"])
s1["VH_over_VV"] = 10 * np.log10(s1["VH_over_VV_gamma0_masked"])

s1["VV_filtered"] = 10 * np.log10(s1["VV_gamma0_masked_filtered"])
s1["VH_filtered"] = 10 * np.log10(s1["VH_gamma0_masked_filtered"])
s1["VH_over_VV_filtered"] = 10 * np.log10(s1["VH_over_VV_gamma0_masked_filtered"])


#### Monthly mean of SAR, monthly median of Sentinel-2

In [ ]:
# group by month
s1_grouped = s1.groupby("time.month")

# make template for monthly xarray dataset
s1_month_template = s1_grouped.mean(dim="time", keep_attrs=True)

# make empty xarray dataset with same structure as template
s1_month = s1_month_template.copy(deep=True)
for var in s1_month.data_vars:
    s1_month[var].values[:] = np.nan


s1_month_filtered = s1_month.copy(deep=True)


In [ ]:
s1_month["mean_vv"] = s1["VV"].groupby("time.month").mean(dim="time", keep_attrs=True)
s1_month["mean_vh"] = s1["VH"].groupby("time.month").mean(dim="time", keep_attrs=True)
s1_month["mean_vh_over_vv"] = (
    s1["VH_over_VV"].groupby("time.month").mean(dim="time", keep_attrs=True)
)

# repeat for the lee filtered variables
s1_month_filtered["mean_vv"] = (
    s1["VV_filtered"].groupby("time.month").mean(dim="time", keep_attrs=True)
)
s1_month_filtered["mean_vh"] = (
    s1["VH_filtered"].groupby("time.month").mean(dim="time", keep_attrs=True)
)
s1_month_filtered["mean_vh_over_vv"] = (
    s1["VH_over_VV_filtered"].groupby("time.month").mean(dim="time", keep_attrs=True)
)


In [ ]:
s1_month["min_vv"] = s1["VV"].groupby("time.month").min(dim="time", keep_attrs=True)
s1_month["min_vh"] = s1["VH"].groupby("time.month").min(dim="time", keep_attrs=True)
s1_month["min_vh_over_vv"] = (
    s1["VH_over_VV"].groupby("time.month").min(dim="time", keep_attrs=True)
)

s1_month["max_vv"] = s1["VV"].groupby("time.month").max(dim="time", keep_attrs=True)
s1_month["max_vh"] = s1["VH"].groupby("time.month").max(dim="time", keep_attrs=True)
s1_month["max_vh_over_vv"] = (
    s1["VH_over_VV"].groupby("time.month").max(dim="time", keep_attrs=True)
)

s1_month["std_vv"] = s1["VV"].groupby("time.month").std(dim="time", keep_attrs=True)
s1_month["std_vh"] = s1["VH"].groupby("time.month").std(dim="time", keep_attrs=True)
s1_month["std_vh_over_vv"] = (
    s1["VH_over_VV"].groupby("time.month").std(dim="time", keep_attrs=True)
)

# repeat for the lee filtered variables
s1_month_filtered["min_vv"] = (
    s1["VV_filtered"].groupby("time.month").min(dim="time", keep_attrs=True)
)
s1_month_filtered["min_vh"] = (
    s1["VH_filtered"].groupby("time.month").min(dim="time", keep_attrs=True)
)
s1_month_filtered["min_vh_over_vv"] = (
    s1["VH_over_VV_filtered"].groupby("time.month").min(dim="time", keep_attrs=True)
)

s1_month_filtered["max_vv"] = (
    s1["VV_filtered"].groupby("time.month").max(dim="time", keep_attrs=True)
)
s1_month_filtered["max_vh"] = (
    s1["VH_filtered"].groupby("time.month").max(dim="time", keep_attrs=True)
)
s1_month_filtered["max_vh_over_vv"] = (
    s1["VH_over_VV_filtered"].groupby("time.month").max(dim="time", keep_attrs=True)
)

s1_month_filtered["std_vv"] = (
    s1["VV_filtered"].groupby("time.month").std(dim="time", keep_attrs=True)
)
s1_month_filtered["std_vh"] = (
    s1["VH_filtered"].groupby("time.month").std(dim="time", keep_attrs=True)
)
s1_month_filtered["std_vh_over_vv"] = (
    s1["VH_over_VV_filtered"].groupby("time.month").std(dim="time", keep_attrs=True)
)


In [ ]:
s1_month.compute()

s1_month_filtered.compute()


In [ ]:
s2_month = s2.groupby("time.month").median("time", keep_attrs=True)


In [ ]:
# delete original s1 to save memory
del s1
del s1_grouped
del s1_month_template
del s2
gc.collect()


#### Compare the monthly Sentinel-2 mean, with and without speckle filtering

A quick visual comparison of the mean monthly composites from Sentinel-1, with and without Lee filtering applied.

In [ ]:
# before and after speckle filtering

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5),
    layout="constrained",
    subplot_kw={"projection": ccrs.epsg(3577)},
)

s1_timestep = s1_month.isel(month=0)
s1_timestep_filtered = s1_month_filtered.isel(month=0)

vmin = float(s1_timestep["mean_vv"].quantile(0.02))
vmax = float(s1_timestep["mean_vv"].quantile(0.98))

vmin_fil = float(s1_timestep_filtered["mean_vv"].quantile(0.02))
vmax_fil = float(s1_timestep_filtered["mean_vv"].quantile(0.98))

im0 = s1_timestep["mean_vv"].plot(
    ax=axes[0],
    cmap="Greys_r",
    robust=True,
    transform=ccrs.epsg(3577),
    add_colorbar=False,
    vmin=vmin,
    vmax=vmax,
)
axes[0].set_title("Monthly mean, no speckle filtering")
gl0 = axes[0].gridlines(draw_labels=True)
gl0.xlines = False
gl0.ylines = False
gl0.right_labels = False
gl0.top_labels = False

im1 = s1_timestep_filtered["mean_vv"].plot(
    ax=axes[1],
    cmap="Greys_r",
    robust=True,
    transform=ccrs.epsg(3577),
    add_colorbar=False,
    vmin=vmin_fil,
    vmax=vmax_fil,
)
axes[1].set_title("Monthly mean, with speckle filtering")
gl1 = axes[1].gridlines(draw_labels=True)
gl1.xlines = False
gl1.ylines = False
gl1.right_labels = False
gl1.top_labels = False


cbar = fig.colorbar(
    im1, ax=axes, orientation="horizontal", fraction=0.05, pad=0.07, aspect=30
)
plt.show()


The assignment of vertical and horizontal bands as red and green, respectively, and the ratio as the blue band, is a common way to visualise SAR data.

In [ ]:
rgb(
    s1_month_filtered,
    bands=["mean_vv", "mean_vh", "mean_vh_over_vv"],
    col="month",
    col_wrap=3,
    size=4,
    robust=True,
)


## prepare data for PCA and do PCA


In [ ]:
# s2_month.compute()


In [ ]:
if "month" in s1_month.dims:
    s1_month = s1_month.rename({"month": "time"})
if "month" in s1_month_filtered.dims:
    s1_month_filtered = s1_month_filtered.rename({"month": "time"})
if "month" in s2_month.dims:
    s2_month = s2_month.rename({"month": "time"})


In [ ]:
x_s1 = sklearn_flatten(s1_month_filtered)
pca_s1 = PCA(n_components=4)


In [ ]:
x_s1_clean = np.where(np.isfinite(x_s1), x_s1, np.nan)
x_s1_clean = np.nan_to_num(x_s1_clean)

pca_s1.fit(x_s1_clean)
print("Relative variance in principal components:", pca_s1.explained_variance_ratio_)


In [ ]:
predict_s1 = pca_s1.transform(x_s1_clean)


In [ ]:
out_s1 = sklearn_unflatten(predict_s1, s1_month_filtered)
out_s1 = out_s1.to_dataset(dim=out_s1.dims[0]).transpose("time", "y", "x")


In [ ]:
rgb(out_s1, bands=[0, 1, 2], col="time", col_wrap=3, size=4)


## Random sampling to build training dataset from land cover 2.0

In [ ]:
lc = lc["level3"].squeeze()
lc = lc.where(lc != 255)  # Mask out no data values


In [ ]:
# Equal stratified random samples
train_points = xr_random_sampling(lc, sampling="equal_stratified_random", n=1000)


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 6), sharey=True)

colour_scheme = get_colour_scheme("level3")
cmap, norm = lc_colourmap(colour_scheme)

# Trim no-data off the colormap
cmap2 = ListedColormap(cmap.colors[:-1])

s2_month[["nbart_red", "nbart_green", "nbart_blue"]].isel(
    time=0
).to_array().plot.imshow(robust=True, ax=ax[0], add_labels=False)
train_points.plot(ax=ax[0], column="class", cmap=cmap2, legend=True, categorical=True)

ax[0].set_title("RGB Sentinel-2 Median, month 0")
ax[0].axes.get_xaxis().set_ticks([])
ax[0].axes.get_yaxis().set_ticks([])

s1_month_filtered[["mean_vv", "mean_vh", "mean_vh_over_vv"]].isel(
    time=0
).to_array().plot.imshow(robust=True, ax=ax[1], add_labels=False)
ax[1].set_title("RGB Sentinel-1 Median, month 0 (VV, VH, VH over VV)")
ax[1].axes.get_xaxis().set_ticks([])
ax[1].axes.get_yaxis().set_ticks([])


im = lc.plot(cmap=cmap, norm=norm, ax=ax[2], add_labels=False, add_colorbar=False)
make_colourbar(fig, ax[2], measurement="level3", labelsize=7, horizontal=False)

ax[2].set_title("DEA Land Cover")
ax[2].axes.get_xaxis().set_ticks([])
ax[2].axes.get_yaxis().set_ticks([])


## Combine original bands, PCA and GLCM to use for random forrest

In [ ]:
train_points


In [ ]:
combined_inputs = xr.merge([s1_month_filtered, out_s1])


In [ ]:
cols = ["class"] + [
    f"{stat}_{band}_{month}"
    for stat in ["mean", "max", "min", "std"]
    for band in ["vv", "vh", "vh_over_vv"]
    for month in range(6, 13)
]

cols_pca = [f"pc{pc}_{month}" for pc in range(1, 5) for month in range(6, 13)]

cols = cols + cols_pca

train_df = pd.DataFrame(columns=cols, index=range(len(train_points)))


In [ ]:
train_df.columns


In [ ]:
for i, (c, g) in enumerate(zip(train_points["class"], train_points.geometry)):
    print(i, end="\r")
    ds_point = combined_inputs.sel(x=g.x, y=g.y, method="nearest")

    # assign response variable (row i, first column, i.e. column index 0)
    train_df.loc[i, "class"] = c

    for col in train_df.columns[1:]:
        var = "_".join(col.split("_")[:-1])
        month = int(col.split("_")[-1])

        train_df.loc[i, col] = ds_point.sel(time=month)[var].data


In [ ]:
train_df.to_csv("lc_samples.csv", index=False)


### Split train/test

In [ ]:
def split_train_test(df, test_size=0.2):
    # apply stratified sampling
    strat_split = StratifiedShuffleSplit(n_splits=1, test_size=test_size)
    for train_index, test_index in strat_split.split(df, df["class"]):
        train = df.loc[train_index].copy()
        test = df.loc[test_index].copy()
        return train, test


In [ ]:
train_final, test_final = split_train_test(train_df)


In [ ]:
predictors = cols[1:]


In [ ]:
# define response and explanatory vars train_final dataset
resp_train = train_final['class'].astype(int).copy()
expl_train = train_final[predictors].copy()

# define response and explanatory for test_final
resp_test = test_final['class'].astype(int).copy()
expl_test = test_final[predictors].copy()

print(f"train count: {len{resp_train.index}}")
print(f"test count: {len{resp_test.index}}")


### Random forest classifier

In [ ]:
rf = RandomForestClassifier(n_jobs=-1)


In [ ]:
# ranges of possible hyperparameters
hyperpar = {
    "n_estimators": randint(low=10, high=100),
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": randint(low=5, high=20),
    "min_samples_split": randint(low=2, high=20),
    "min_samples_leaf": randint(low=1, high=20),
    "max_features": randint(low=1, high=len(predictors)),
    "bootstrap": [True, False],
}


In [ ]:
## N.B.: ideally a random state/random seed should be set
rf_rand_search = RandomizedSearchCV(
    rf,
    param_distributions=hyperpar,
    n_iter=200,
    scoring="f1_weighted",
    cv=5,
    verbose=1,
    n_jobs=-1,
)

rf_rand_search.fit(expl_train, resp_train)


In [ ]:
# best model and score of best model
rf_rand_search.best_params_, rf_rand_search.best_score_


In [ ]:
hyperparameters = [
    "n_estimators",
    "max_depth",
    "min_samples_split",
    "min_samples_leaf",
    "max_features",
]

for hyperparam_name in hyperparameters:

    hyperparam_values = rf_rand_search.cv_results_["param_" + hyperparam_name]
    mean_scores = rf_rand_search.cv_results_["mean_test_score"]

    # extract all scores for each hyperparameter value
    hyperparam_tot_scores = dict()
    for value, score in zip(hyperparam_values, mean_scores):
        if value not in hyperparam_tot_scores:
            hyperparam_tot_scores[value] = [score]
        else:
            hyperparam_tot_scores[value].append(score)

    # mean scores for each hyperparameter value
    sorted_hyperparams = sorted(hyperparam_tot_scores.keys())
    averaged_scores = [
        np.nanmean(hyperparam_tot_scores[val]) for val in sorted_hyperparams
    ]

    plt.figure(figsize=(40, 6))
    plt.plot(sorted_hyperparams, averaged_scores, marker="o", linestyle="-")

    plt.xlim([min(sorted_hyperparams), max(sorted_hyperparams)])
    plt.ylabel("Mean Score")
    plt.title(f"{hyperparam_name}")
    plt.grid(True)

    plt.show()


In [ ]:
hyperpar = {
    "n_estimators": [30, 40, 50],
    "criterion": ["log_loss"],
    "max_depth": [10, 11, 12],
    "min_samples_split": [5, 9, 15],
    "min_samples_leaf": [3, 4, 5],
    "max_features": [20, 40, 60],
    "bootstrap": [True],
}

# search all combinations of above values
rf_grid_search = GridSearchCV(
    rf, param_grid=hyperpar, scoring="f1_weighted", cv=5, verbose=1, n_jobs=-1
)

rf_grid_search.fit(expl_train, resp_train)


In [ ]:
rf_grid_search.best_params_, rf_grid_search.best_score_


In [ ]:
# best model
rf_best = rf_grid_search.best_estimator_


In [ ]:
dump(rf_best, f"RF_lc_classifier.joblib")


### Feature importance

In [ ]:
feat_importances = rf_best.feature_importances_
rf_importance = zip(predictors, feat_importances)

plt.figure(figsize=(15, 15))
plt.barh(predictors, feat_importances)
plt.xlabel("Feature importance")
plt.ylabel("Feature")
plt.show()


In [ ]:
# true labels and predictions
y_true = resp_test
y_pred = rf_best.predict(expl_test[predictors])

cm = confusion_matrix(y_true, y_pred)

# show percentages (recall)
cm_normalised = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(cm_normalised, annot=True, fmt=".2%", cmap="Blues", cbar=False, ax=ax)
ax.set_xlabel("Predicted Labels")
ax.set_ylabel("True Labels")
ax.set_title("Confusion Matrix (Percentage)")

# add counts in the cells
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j + 0.5, i + 0.5, f"{cm[i, j]}\n", ha="center", va="bottom", color="black"
        )

plt.show()

# metrics
accuracy = accuracy_score(y_true, y_pred)
f1_weighted = f1_score(y_true, y_pred, average="weighted")
precision_weighted = precision_score(y_true, y_pred, average="weighted")
recall_weighted = recall_score(y_true, y_pred, average="weighted")

print(f"Accuracy: {accuracy:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")
print(f"Weighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")


In [ ]:
# classification summary
report = classification_report(y_true, y_pred, output_dict=True)

report_df = pd.DataFrame(report).transpose()

print(report_df)


### Plot on map 

In [ ]:
# lc_samples = train_points_equal_stratified_random.copy()
# lc_samples["x"] = lc_samples.geometry.x
# lc_samples["y"] = lc_samples.geometry.y


In [ ]:
# def extract_features_at_points(xr_data, x_coords, y_coords):
#     features = []
#     for xi, yi in zip(x_coords, y_coords):
#         vals = []
#         for var in xr_data.data_vars:
#             vals.extend(xr_data[var].sel(x=xi, y=yi, method="nearest").values.flatten())
#         features.append(vals)
#     return np.array(features)


In [ ]:
# lc_samples = train_points_equal_stratified_random.copy()
# lc_samples["x"] = lc_samples.geometry.x
# lc_samples["y"] = lc_samples.geometry.y

# # Flatten the combined_inputs grid for ML
# X_grid_full = sklearn_flatten(combined_inputs)

# # Get the x, y coordinates of the grid
# x_coords = combined_inputs["x"].values
# y_coords = combined_inputs["y"].values


# # Build a mapping from (x, y) to flattened index
# def get_flat_index(xi, yi, x_coords, y_coords):
#     x_idx = np.abs(x_coords - xi).argmin()
#     y_idx = np.abs(y_coords - yi).argmin()
#     return y_idx * len(x_coords) + x_idx


# # Get indices for training points
# indices = [
#     get_flat_index(xi, yi, x_coords, y_coords)
#     for xi, yi in zip(lc_samples["x"], lc_samples["y"])
# ]

# # Extract features for training points from the flattened grid
# X = X_grid_full[indices]
# y = lc_samples["class"].values

# # Train/validation split
# X_train, X_val, y_train, y_val = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42
# )

# # Train random forest
# rf = RandomForestClassifier(n_estimators=100, random_state=42)
# rf.fit(X_train, y_train)

# val_score = rf.score(X_val, y_val)
# print(f"Validation accuracy: {val_score:.3f}")

# # Predict over all data arrays
# y_pred_grid = rf.predict(X_grid_full)

# # Unflatten to raster
# raster_pred = sklearn_unflatten(y_pred_grid, combined_inputs)
# raster_pred = xr.DataArray(
#     raster_pred,
#     dims=combined_inputs.dims,
#     coords=combined_inputs.coords,
#     name="rf_class",
# )


# # Plot
# raster_pred.plot.imshow(x="x", y="y", col="time", cmap="tab10", col_wrap=3, size=4)
# plt.show()
